# Vision-Language Models (VLM)

> In the previous part we looked at CoT and Thinking models, focusing on making LLMs better at reasoning within text. Now we switch to a different question: images are not text, so why can an LLM also "see and describe pictures"?
>
> In this section, we build a minimal VLM from scratch. We start by cutting an image into patches, turning each patch into a vector (a visual token), and finally concatenating visual tokens with text tokens before feeding everything into a Transformer.

You can phrase the core confusion in one sentence: LLMs only understand tokens, but images are arrays of pixels. How does a photo of a cat become tokens that an LLM can read?

A **Vision-Language Model (VLM)** is, by definition, a model that can process both images and text, and respond in natural language. For example, given a photo of a menu and the question "what is the cheapest drink?", the model must first understand the image, then answer in language.

In this section we will not call any pre-trained model. Instead we build a minimal VLM from scratch. The path is straightforward:

1. Cut the image into small blocks, called patches.
2. Turn each patch into a vector, called a visual token.
3. Concatenate the visual tokens with text tokens and feed them into a Transformer.

It sounds like "giving an LLM a pair of eyes", but these eyes do not look at pixels directly. Instead, they first translate pixels into embeddings that the LLM already knows how to process.

## 1. Why Can't LLMs Read Images Directly?

Let us define two terms first.

A **token** is the smallest unit a model processes at one time. In text, a token might be a character, a word, or part of a word. For example, the sentence "the cat sat" might be turned into several integer IDs by a tokenizer.

An **embedding** is the vector representation of a token. For example, token id `12` is just a number, but during computation the model looks it up in a table and turns it into a vector like `[0.3, -0.1, ...]`.

The problem is: an image is not a sequence of tokens. It is a two-dimensional grid of pixels.

```
Text: "a cat sitting on a mat"
  -> Tokenizer -> [12, 45, 78, 3, 90, 23]
  -> Embedding -> [6 vectors]

Image: 224 x 224 pixel photo of a cat
  -> ???       -> LLM does not know how to handle this yet
```

So the fundamental question for VLMs is: **how do we turn an image into a sequence of embeddings too?**

The simplest answer: cut the image into many small blocks, and treat each block as a "visual word". This way, the image goes from a 2D grid to a 1D sequence.

## 2. Patchify: Cutting the Image into Small Blocks

**Patchify** means "cutting an image into patches". A patch is a small block of the image.

For example, a 224 x 224 image with a patch size of 16 x 16 produces 14 patches per row and 14 patches per column, for a total of 14 x 14 = 196 patches.

```
Original image: 224 x 224 pixels
patch size: 16 x 16

224 / 16 = 14
14 x 14 = 196 patches

+--+--+--+--+
|  |  |  |  |
+--+--+--+--+
|  |cat|  |  |  <- the cat's face might fall in a few patches
+--+--+--+--+
|  |  |  |  |
+--+--+--+--+
```

Why not turn the whole image into one token? Because a single token is too coarse — the model would have no way to know where the cat's face, text, or background are. Cutting into patches at least preserves "local region" information.

Let us use `unfold` to manually inspect the shape after patchifying, without any vision model.

In [ ]:
# First compute the shape by hand, then verify with code
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

IMG_SIZE = 224
PATCH_SIZE = 16

# Simulate an RGB image: [color channels, height, width]
fake_image = torch.randn(3, IMG_SIZE, IMG_SIZE)

print("Original image shape:", fake_image.shape)
print("Explanation: 3 color channels, each channel is a 224 x 224 grid")

# unfold slides a window along height and width. stride=16 means no overlap.
patches = fake_image.unfold(1, PATCH_SIZE, PATCH_SIZE)
patches = patches.unfold(2, PATCH_SIZE, PATCH_SIZE)

print("\nShape after patchifying:", patches.shape)
print("Explanation: [3 channels, 14 row-patches, 14 col-patches, 16 height, 16 width]")

num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
patch_dim = 3 * PATCH_SIZE * PATCH_SIZE
patches_flat = patches.permute(1, 2, 0, 3, 4).reshape(num_patches, patch_dim)

print("\nFlattened shape:", patches_flat.shape)
print(f"Key observation: one image becomes {num_patches} patches")
print(f"Each patch has {patch_dim} numbers, which is 3 x 16 x 16")

## 3. Patch Embedding: Turning Each Block into a Vector

Now each patch has 768 numbers, because $3 \times 16 \times 16 = 768$.

But in an LLM, each text token's embedding is typically a fixed dimension such as 512, 768, or 4096. If visual patches want to mix into the LLM, they must become vectors of the same dimension.

This step is called **Patch Embedding**: projecting the pixel numbers of a patch into a `d_model`-dimensional vector.

```
One patch:       768 pixel numbers
Patch Embedding: Linear(768 -> d_model)
Output:          1 visual token of dimension d_model

Text token:      Embedding lookup -> 1 text token of dimension d_model

Same dimension, so they can be concatenated later.
```

In practice, ViT often uses `Conv2d` to perform "cut into patches + linear projection" in one step. The convolution here is not for detecting edges — it is simply a convenient way to implement patch embedding.

In [ ]:
class PatchEmbedding(nn.Module):
    """Cut an image into patches and map each patch to a visual token."""
    def __init__(self, img_size=224, patch_size=16, in_channels=3, d_model=512):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        # kernel_size=patch_size: look at one patch at a time
        # stride=patch_size: the next window lands exactly on the next patch
        self.proj = nn.Conv2d(
            in_channels,
            d_model,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        """
        Args:
            x: [batch, 3, 224, 224] image tensor
        Returns:
            [batch, num_patches, d_model] visual tokens
        """
        x = self.proj(x)      # [batch, d_model, 14, 14]
        x = x.flatten(2)      # [batch, d_model, 196]
        x = x.transpose(1, 2) # [batch, 196, d_model]
        return x

patch_emb = PatchEmbedding(img_size=224, patch_size=16, d_model=512)
dummy_img = torch.randn(2, 3, 224, 224)
visual_tokens = patch_emb(dummy_img)

print("Input image:", dummy_img.shape)
print("Visual tokens:", visual_tokens.shape)
print("Key observation: each image in the batch has 196 visual tokens, each of dimension 512")

## 4. How Do Visual Tokens Merge with Text Tokens?

At this point, the image has become `[batch, 196, d_model]`. Text also becomes `[batch, text_len, d_model]` through an embedding layer.

Since the last dimension of both is `d_model`, the simplest fusion method is: **concatenate along the sequence length dimension**.

```
Visual tokens: [vis_1] [vis_2] ... [vis_196]
Text tokens:   [please] [describe] [this] [image]

After concatenation:
[vis_1] [vis_2] ... [vis_196] [please] [describe] [this] [image]
```

This is the intuition behind Visual Token schemes like LLaVA: the LLM does not need to know that "the first 196 positions are from an image". It only sees a sequence of embeddings. Training will gradually teach it that the vectors at those special positions come from an image.

In [ ]:
# Simulate concatenation of visual tokens and text tokens
d_model = 512
text_vocab_size = 1000

# Assume the tokenizer turns "please describe this image" into 5 ids
text_ids = torch.tensor([[5, 12, 78, 3, 90]])
text_embedding = nn.Embedding(text_vocab_size, d_model)
text_tokens = text_embedding(text_ids)

visual_tokens = torch.randn(1, 196, d_model)
combined = torch.cat([visual_tokens, text_tokens], dim=1)

print("Visual tokens:", visual_tokens.shape)
print("Text tokens:", text_tokens.shape)
print("After concatenation:", combined.shape)
print("Key observation: 196 image tokens + 5 text tokens = 201 tokens")

## 5. The Missing Piece: A Projector to Translate from Vision Space to Language Space

The `PatchEmbedding` above was a teaching version that directly projected pixels into `d_model`. Real VLMs usually add an extra step:

1. A Vision Encoder first encodes the image into visual features.
2. A Projector then maps the visual features into the LLM's embedding space.

Why do we need a Projector? Because the Vision Encoder and the LLM are two separate models — their vector spaces are not naturally aligned.

An analogy: the Vision Encoder speaks a "visual dialect" and the LLM speaks a "language dialect". The Projector acts as a translator, converting visual features into embeddings that the LLM can understand.

```
Image -> Vision Encoder -> visual features [B, 196, d_vis]
                       -> Projector -> language space [B, 196, d_llm]
Text  -> Token Embedding -----------------------> [B, T, d_llm]

Final concatenation: [B, 196 + T, d_llm]
```

LLaVA v1 uses a single Linear layer; LLaVA 1.5 commonly uses a two-layer MLP. Below we implement a small Projector to see the shape transformation.

In [ ]:
class LLaVAProjector(nn.Module):
    """Map visual features into the LLM's embedding space."""
    def __init__(self, vis_dim=1024, llm_dim=4096):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(vis_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )

    def forward(self, visual_features):
        """
        Args:
            visual_features: [batch, num_patches, vis_dim]
        Returns:
            [batch, num_patches, llm_dim]
        """
        return self.mlp(visual_features)

projector = LLaVAProjector(vis_dim=1024, llm_dim=4096)
vision_features = torch.randn(2, 196, 1024)
llm_space_features = projector(vision_features)

print("Vision Encoder output:", vision_features.shape)
print("Projector output:", llm_space_features.shape)
print("Key observation: the number of patches stays the same, only each token's dimension is aligned to the LLM")

## 6. Three Mainstream VLM Architectures

Now you understand the most straightforward Visual Token approach. Industry solutions can be grouped into three categories.

| Approach | How images enter the LLM | Does the LLM structure change? | Representative model |
|:---|:---|:---:|:---|
| Visual Token | Image tokens concatenated directly into the sequence | No | LLaVA |
| Cross-Attention | Text tokens query visual features | Yes | Flamingo |
| Q-Former | A small number of queries compress visual features | Usually no | BLIP-2 |

### 6.1 Visual Token: The Most Intuitive

The image becomes 196 tokens, placed directly in front of the text. The advantage is that the main LLM body does not need to change, making it the fastest to implement in practice. The disadvantage is also clear: the sequence becomes longer, making attention computation more expensive.

### 6.2 Cross-Attention: Letting Text Actively "Look at the Image"

Self-Attention is about tokens within a sequence attending to each other: Q, K, V all come from the same sequence.

Cross-Attention is about text attending to images: Q comes from text, K and V come from visual features.

The only difference in the formula is the source of K and V:

$$
\text{CrossAttn}(X_{text}, X_{vis}) =
\text{softmax}\left(\frac{Q_{text}K_{vis}^T}{\sqrt{d_k}}\right)V_{vis}
$$

Flamingo also adds a `tanh(alpha)` gating mechanism. At initialization `alpha=0`, so the visual branch has almost no effect on the original LLM at first. During training the gate gradually opens, and the model slowly learns to use image information.

### 6.3 Q-Former: Compress First, Then Pass to the LLM

Q-Former's goal is to reduce the number of visual tokens. For example, 196 patch tokens can be compressed into 32 tokens through 32 learnable queries. This makes inference faster, but the architecture is also more complex.

Below we use code to compare the computational cost intuition of Visual Token vs. Cross-Attention.

In [ ]:
class CrossAttention(nn.Module):
    """Cross-Attention: Q from text, K/V from images."""
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.scale = self.head_dim ** -0.5

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, text_hidden, image_features):
        """
        Args:
            text_hidden: [batch, text_len, d_model]
            image_features: [batch, num_visual, d_model]
        Returns:
            [batch, text_len, d_model]
        """
        batch, text_len, d_model = text_hidden.shape
        num_visual = image_features.shape[1]

        query = self.q_proj(text_hidden)
        key = self.k_proj(image_features)
        value = self.v_proj(image_features)

        query = query.view(batch, text_len, self.n_heads, self.head_dim)
        query = query.transpose(1, 2)
        key = key.view(batch, num_visual, self.n_heads, self.head_dim)
        key = key.transpose(1, 2)
        value = value.view(batch, num_visual, self.n_heads, self.head_dim)
        value = value.transpose(1, 2)

        scores = (query @ key.transpose(-2, -1)) * self.scale
        weights = F.softmax(scores, dim=-1)
        out = weights @ value
        out = out.transpose(1, 2).reshape(batch, text_len, d_model)
        return self.out_proj(out)


class GatedCrossAttentionBlock(nn.Module):
    """A small Cross-Attention module with tanh gating."""
    def __init__(self, d_model=512, n_heads=8):
        super().__init__()
        self.cross_attn = CrossAttention(d_model, n_heads)
        self.norm = nn.LayerNorm(d_model)
        self.alpha = nn.Parameter(torch.zeros(1))

    def forward(self, text_hidden, image_features):
        gate = torch.tanh(self.alpha)
        x = text_hidden + gate * self.cross_attn(text_hidden, image_features)
        return self.norm(x)


batch = 1
text_len = 50
num_visual = 196
d_model = 512

text_hidden = torch.randn(batch, text_len, d_model)
image_features = torch.randn(batch, num_visual, d_model)

visual_token_len = text_len + num_visual
visual_token_cost = visual_token_len ** 2
cross_attn_cost = text_len ** 2 + text_len * num_visual

block = GatedCrossAttentionBlock(d_model=d_model, n_heads=8)
out = block(text_hidden, image_features)

print("Visual Token sequence length:", visual_token_len)
print("Visual Token attention approx. cost:", visual_token_cost)
print("Cross-Attention output shape:", out.shape)
print("Cross-Attention approx. cost:", cross_attn_cost)
print("Gate initial value tanh(alpha):", torch.tanh(block.alpha).item())
print("Key observation: Cross-Attention does not lengthen the text sequence, but requires modifying the LLM structure")

## 7. Why Are Images So "Expensive"?

A 224 x 224 image, when cut into 16 x 16 patches, produces 196 visual tokens.

A sentence like "please describe this image" might only be a few tokens. In other words, within a VLM, the thing that really takes up context length is often not the text, but the image.

This has two consequences:

1. Inference is slower, because attention complexity is approximately $O(n^2)$.
2. Multiple images are even more expensive, because each image can bring hundreds of visual tokens.

Can we reduce the number of visual tokens? Yes, but at the cost of losing detail. Larger patches mean fewer tokens; smaller patches mean more detail but more tokens.

In [ ]:
# Intuition: larger patches mean fewer visual tokens
for patch_size in [8, 14, 16, 32]:
    tokens_per_side = 224 // patch_size
    num_tokens = tokens_per_side ** 2
    print(f"patch_size={patch_size:2d}: {tokens_per_side:2d} x {tokens_per_side:2d}",
          f"= {num_tokens:4d} visual tokens")

print("\nKey observation: patch=32 has only 49 tokens, but detail is coarser")
print("Key observation: patch=8 has 784 tokens, more detail, but slower inference")

## 8. Training a VLM: Why Freeze?

VLMs are usually not trained from scratch on all three large modules. They reuse a pre-trained Vision Encoder and LLM, then focus training on the connection in between.

First, consider the three components:

```
Vision Encoder: already knows how to look at images, e.g. CLIP / SigLIP / ViT
Projector:      trained from scratch, responsible for translating visual features into LLM space
LLM:            already knows how to speak, reason, and generate text
```

Why not train everything together?

- Too many parameters, high GPU memory cost.
- Not enough data; easy to damage the pre-trained ability to see images or generate language.
- The Projector is the component that most needs to learn from scratch.

So the classic LLaVA-style approach usually has two stages. Note that "freezing the Vision Encoder" is a common engineering choice in the LLaVA pipeline, not an eternal rule for all VLMs; if data and compute are sufficient, one can also fine-tune the vision tower or do multi-stage unfreezing.

| Stage | What is trained | What is frozen | Goal |
|:---|:---|:---|:---|
| Stage 1: Alignment training | Projector | Vision Encoder + LLM | Make visual features resemble LLM embeddings |
| Stage 2: Instruction tuning | Projector + LLM | Vision Encoder | Learn to answer questions about images |

"Freezing" here means setting `requires_grad = False` on the parameters. They still participate in forward computation, but will not be updated by gradients.

In [ ]:
# Use three small modules to simulate the VLM freezing strategy
vision_encoder = nn.Linear(1024, 1024)
projector = LLaVAProjector(vis_dim=1024, llm_dim=4096)
llm_head = nn.Linear(4096, 1000)

def set_trainable(module, trainable):
    """Set a module to be trainable or frozen."""
    for param in module.parameters():
        param.requires_grad = trainable

def count_trainable(module):
    """Count the number of trainable parameters in a module."""
    return sum(param.numel() for param in module.parameters() if param.requires_grad)

def show_stage(stage):
    """Show the freezing result for a training stage."""
    if stage == 1:
        set_trainable(vision_encoder, False)
        set_trainable(projector, True)
        set_trainable(llm_head, False)
    else:
        set_trainable(vision_encoder, False)
        set_trainable(projector, True)
        set_trainable(llm_head, True)

    print(f"Stage {stage}")
    print("  Vision Encoder trainable params:", count_trainable(vision_encoder))
    print("  Projector trainable params:     ", count_trainable(projector))
    print("  LLM Head trainable params:      ", count_trainable(llm_head))

show_stage(1)
print()
show_stage(2)
print("\nKey observation: Vision Encoder is frozen in both stages, Projector is trained in both stages")

## 9. Engineering Details: Special Tokens, Position Encodings, Multi-Resolution

The main flow above is enough for building a minimal VLM. But real models also face several engineering challenges.

**Problem 1: How does the LLM know where the image starts?**

Some models wrap visual content with special tokens:

```
<image_start> [vis_1] [vis_2] ... [vis_196] <image_end> please describe this image
```

The early LLaVA approach could simply place visual tokens at the front; with multiple images or multi-turn dialogue, special tokens become clearer.

**Problem 2: What position encoding do visual tokens use?**

Images are inherently two-dimensional, but once concatenated into the LLM they become a one-dimensional sequence. The simplest approach is to number them left-to-right, top-to-bottom:

```
Row 0: patch 0, patch 1, ..., patch 13
Row 1: patch 14, patch 15, ..., patch 27
```

This works but loses some 2D spatial structure. More sophisticated models introduce 2D position encodings or dynamic resolution strategies.

**Problem 3: What about high-resolution images?**

If a 4K image is forcibly resized to 224 x 224, small text and fine details are lost. Methods like AnyRes keep one thumbnail, then cut the large image into multiple local sub-images and encode each separately. The trade-off is a significant increase in visual token count.

## 10. A Minimal VLM Implementation

Now let us connect all the components above. This version is very small, but the data flow is complete:

1. `PatchEmbedding` turns the image into visual tokens.
2. `text_embedding` turns text ids into text tokens.
3. `torch.cat` concatenates the two types of tokens into one sequence.
4. `TransformerEncoder` processes the entire sequence.
5. `lm_head` predicts the next-token distribution at each position.

This is not a truly conversational VLM, since we have not trained it and have not connected a real tokenizer. But it can help you confirm: the core of a VLM is not magic, but a very ordinary tensor pipeline.

In [ ]:
class MiniVLM(nn.Module):
    """A minimal Visual Token VLM."""
    def __init__(self, text_vocab_size=100, d_model=128, num_heads=2, num_layers=2):
        super().__init__()
        self.vision = PatchEmbedding(
            img_size=224,
            patch_size=16,
            in_channels=3,
            d_model=d_model,
        )
        self.text_embedding = nn.Embedding(text_vocab_size, d_model)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=num_heads,
            dim_feedforward=4 * d_model,
            batch_first=True,
        )
        self.layers = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, text_vocab_size)

    def forward(self, images, text_ids):
        """
        Args:
            images: [batch, 3, 224, 224]
            text_ids: [batch, text_len]
        Returns:
            [batch, 196 + text_len, vocab_size]
        """
        visual_tokens = self.vision(images)
        text_tokens = self.text_embedding(text_ids)
        tokens = torch.cat([visual_tokens, text_tokens], dim=1)
        hidden = self.layers(tokens)
        hidden = self.norm(hidden)
        return self.lm_head(hidden)

vlm = MiniVLM(text_vocab_size=100, d_model=128, num_heads=2, num_layers=2)

dummy_images = torch.randn(1, 3, 224, 224)
dummy_text = torch.randint(0, 100, (1, 10))
logits = vlm(dummy_images, dummy_text)

print("Input image:", dummy_images.shape)
print("Input text:", dummy_text.shape)
print("Output logits:", logits.shape)
print("Key observation: 196 visual positions + 10 text positions = 206 output positions")
print("Total parameters:", f"{sum(param.numel() for param in vlm.parameters()):,}")

## Summary

These are the key points you should take away from this section:

- [ ] VLM definition: a model that processes both images and text, and outputs results in language.
- [ ] Images cannot directly enter an LLM; they must first be turned into a sequence of visual tokens.
- [ ] Patchify can cut a 224 x 224 image into 196 patches of 16 x 16; 16 x 16 is one classic choice in ViT, though real models also commonly use 14, 16, 32, or dynamic patching.
- [ ] Patch Embedding turns each patch into a `d_model`-dimensional vector.
- [ ] The Visual Token approach concatenates visual tokens and text tokens directly; the main LLM body does not need to change.
- [ ] The Cross-Attention approach lets text tokens query visual features; the sequence does not get longer, but the structure must change.
- [ ] The Projector is the translator from "vision space" to "LLM space".
- [ ] Training a VLM commonly uses two stages: first train the Projector for alignment, then train Projector + LLM for instruction tuning.

| Concept | One-line explanation |
|:---|:---|
| Patch | A small block of the image, e.g. 16 x 16 pixels; patch size varies across real models |
| Patch Embedding | The layer that turns a patch into a vector |
| Visual Token | The token vector corresponding to an image patch |
| Projector | Maps visual features into the LLM embedding space |
| Cross-Attention | Attention where Q comes from text and K/V come from images |
| Q-Former | Uses a small number of queries to compress many visual features into few tokens |

**One-sentence summary**: A VLM does not let the LLM look at pixels directly. Instead, it first translates the image into a sequence of embeddings, then lets the LLM read those embeddings just like it reads text. Closed-source models like GPT-4V have not published their complete visual architecture, so they cannot be taken as evidence for any specific open-source approach. References: [ViT documentation](https://huggingface.co/docs/transformers/main/model_doc/vit), [LLaVA paper](https://arxiv.org/abs/2304.08485), [GPT-4V system card](https://cdn.openai.com/papers/GPTV_System_Card.pdf).

## Exercises

When doing exercises, you can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

### Exercise 1: Change the Patch Size

Change `patch_size` from 16 to 32 and observe how the number of visual tokens changes.

Hint: `num_patches = (img_size // patch_size) ** 2`.

```python
patch_emb_32 = PatchEmbedding(img_size=224, patch_size=____, d_model=128)
assert patch_emb_32.num_patches == ____
print("Passed: you now know that larger patches mean fewer visual tokens.")
```

### Exercise 2: Implement a Single-Layer Projector

Write an `nn.Linear` that projects 1024-dimensional visual features into a 4096-dimensional LLM space.

Hint: the input shape is `[2, 196, 1024]`, and the last dimension of the output should become 4096.

```python
simple_projector = nn.Linear(____, ____)
x = torch.randn(2, 196, 1024)
y = simple_projector(x)
assert y.shape == (2, 196, 4096)
print("Passed: you have completed a dimension alignment from vision space to language space.")
```

### Exercise 3: Calculate Two-Stage Training Ratios

Assume the Vision Encoder has 300M parameters, the LLM has 7B parameters, and the Projector has 8M parameters. Calculate how many parameters are being trained in Stage 1 and Stage 2 respectively.

Hint: Stage 1 only trains the Projector; Stage 2 trains the Projector + LLM.

```python
vision_params = 300_000_000
llm_params = 7_000_000_000
projector_params = 8_000_000

stage1_trainable = ____
stage2_trainable = ____

assert stage1_trainable == 8_000_000
assert stage2_trainable == 7_008_000_000
print("Passed: you now understand why VLMs use staged freezing and training.")
```